In [31]:
# imports
import os
import glob
from dotenv import load_dotenv

import numpy as np

from sklearn.manifold import TSNE

from langchain.document_loaders import DirectoryLoader
from langchain.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.schema import Document
from langchain.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI

import plotly.graph_objects as go

In [36]:
# initialization
load_dotenv(override=True)
embeddings = OpenAIEmbeddings()

In [32]:
# config
MODEL = "gpt-4o-mini"
db_name = "vector_db"
folders = glob.glob("knowledge-base/*")
text_loader_kwargs = {"encoding": "utf-8"}

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

In [25]:
documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs=text_loader_kwargs)
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

len(documents)

31

In [23]:
documents[24]

Document(metadata={'source': 'knowledge-base/contracts/Contract with Roadway Insurance Inc. for Carllm.md', 'doc_type': 'contracts'}, page_content="# Contract with Roadway Insurance Inc. for Carllm\n\n---\n\n## Terms\n\n1. **Agreement Effective Date**: This contract is effective as of January 1, 2025.\n2. **Duration**: This agreement will remain in effect for a term of 12 months, concluding on December 31, 2025.\n3. **Subscription Type**: Roadway Insurance Inc. agrees to subscribe to the **Professional Tier** of Carllm, at a cost of $2,500/month, totaling $30,000 for the duration of this contract.\n4. **Payment Terms**: Payments are due on the first of each month. Late payments will incur a penalty of 1.5% per month.\n5. **Termination Clause**: Either party may terminate this agreement with 30 days' written notice prior to the end of the term. If terminated early, fees will be calculated on a pro-rata basis.\n\n---\n\n## Renewal\n\n1. **Automatic Renewal**: This agreement will automati

In [26]:
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)
len(chunks)

Created a chunk of size 1088, which is longer than the specified 1000


123

In [28]:
chunks[6]

Document(metadata={'source': 'knowledge-base/products/Homellm.md', 'doc_type': 'products'}, page_content='- **Basic Tier:** Starting at $5,000/month for small insurers with basic integration features.\n- **Standard Tier:** Starting at $10,000/month for medium-sized insurers including advanced analytics and reporting tools.\n- **Enterprise Tier:** Custom pricing for large insurance companies that require full customization, dedicated support, and additional features, such as enterprise-grade security and compliance.\n\nAll tiers include a comprehensive training program and ongoing updates to ensure optimal performance.\n\n## Roadmap\nThe development roadmap for Homellm includes the following key milestones:')

In [29]:
doc_types = set(chunk.metadata["doc_type"] for chunk in chunks)
print(f"Document types found: {', '.join(doc_types)}")

Document types found: contracts, company, employees, products


In [30]:
for chunk in chunks:
    if "CEO" in chunk.page_content:
        print(chunk.page_content)
        print("___________________________________________")

# Avery Lancaster

## Summary
- **Date of Birth**: March 15, 1985  
- **Job Title**: Co-Founder & Chief Executive Officer (CEO)  
- **Location**: San Francisco, California  

## Insurellm Career Progression
- **2015 - Present**: Co-Founder & CEO  
  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise that have catapulted the company into the mainstream insurance market.  

- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  
  Before launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking insurance products aimed at the tech sector.
___________________________________________
3. **Regular Updates:** Insurellm will offer ongoing updates and enhancements to the Homellm platform, including new features and security impro

In [37]:
# create vector database
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vector database created with {vectorstore._collection.count()} vectors.")

Vector database created with 123 vectors.


In [66]:
collection = vectorstore._collection
sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"Each embedding has {dimensions} dimensions.")

Each embedding has 1536 dimensions.


In [80]:
result = collection.get(include=["embeddings", "documents", "metadatas"])
vectors = np.array(result["embeddings"])
documents = result["documents"]
doc_types = [metadata["doc_type"] for metadata in result["metadatas"]]
colors = [["blue", "green", "red", "orange"][["products", "employees", "contracts", "company"].index(t)] for t in doc_types]

In [81]:
dict(size=5, color=colors, opacity=0.8)

{'size': 5,
 'color': ['orange',
  'orange',
  'orange',
  'blue',
  'blue',
  'blue',
  'blue',
  'blue',
  'blue',
  'blue',
  'blue',
  'blue',
  'blue',
  'blue',
  'blue',
  'blue',
  'blue',
  'blue',
  'blue',
  'blue',
  'blue',
  'blue',
  'blue',
  'blue',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'green',
  'red',
  'red',
  'red',
  'red',
  'red',
  'red',
  'red',
  'red',
  'red',
  'red',
  'red',
  'red',
  'red',
  'red',
  'red',
  'red',
  'red',
  'red',
  'red',
  'red',
  'red',
  'red',
  'red',
  'red',


In [89]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo="text"
)])

fig.update_layout(
    title="2D Chroma Vector Store Visualization",
    scene=dict(xaxis_title="x", yaxis_title="y"),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [90]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo="text"
)])

fig.update_layout(
    title="3D Chroma Vector Store Visualization",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z"),
    width=900,
    height=700,
    margin=dict(r=20, b=10, l=10, t=40)
)
fig.show()